# Neural Networks and Ensembles
**Jennifer Eigo - University of Connecticut - Dept. of Operations and Information Management**

-------------------------------------
Neural network models can pick up complex patterns in the data which yields very accurate results. However because of this complexity they are known as a black box model.  We don't get to easily learn direct variable impact from this model.  But when you don't require explainability, this is a great choice!  We will also build a simple ensemble model to show how we can combine outputs.  

# Environment Setup

In [ ]:
# import modules

import pandas as pd # for data viz and wrangling
import numpy as np # for 'numeric python'
import matplotlib.pyplot as plt # for data viz (more complex than pylab)
import seaborn as sns
from pylab import * # for data viz (import * means 'import all of the functions')

from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from sklearn import metrics
from scipy import stats
import statsmodels.api as sm


In [ ]:
# mount your google drive
from google.colab import drive
drive.mount('/content/drive')

#Load the Data - ToyotaCorolla1000

In [ ]:
# read data
# on the lefthand side, navigate to your data and copy the path
df = pd.read_csv('/content/drive/MyDrive/OPIM 5604 Python/Module 10/ToyotaCorolla1000.csv')

In [ ]:
# shape
# shows how many rows and columns
# this sample has 1000 rows and 10 columns
df.shape

In [ ]:
# Preview
print(df.head())

In [ ]:
# Create dummy variables for 'Fuel Type'
df = pd.get_dummies(df, columns=['Fuel Type'], drop_first=True)

# Display the first few rows to see the changes
print(df.head())

Before we model, we need to partition the data.

In [ ]:
from sklearn.model_selection import train_test_split

# Define the target variable
target = 'Price'
features = df.drop(target, axis=1)
target_variable = df[target]

# Split the data into 80% training and 20% test
X_train, X_test, y_train, y_test = train_test_split(features, target_variable, test_size=0.2, random_state=42)

# Split the 80% training data into 50% training and 30% validation
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.375, random_state=42) # 0.375 * 0.8 = 0.3

# Print the shapes of the resulting datasets
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Test set shape:", X_test.shape, y_test.shape)

#Neural Network

Let's build a neural network model.  First we standardize the predictors then we fit the model.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error # Import mean_squared_error to calculate MSE first

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


# Build the neural network model
nn_model = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=2000, random_state=42, learning_rate_init=0.01)

# Train the model
nn_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
y_pred_val = nn_model.predict(X_val_scaled)

# Evaluate the model
mse = mean_squared_error(y_val, y_pred_val)
rmse = np.sqrt(mse) # Calculate RMSE
r2 = r2_score(y_val, y_pred_val)

print(f"Root Mean Squared Error on validation set: {rmse}") # Print RMSE
print(f"R-squared on validation set: {r2}")

Now lets check its performance on each partition.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Make predictions on all three sets
y_pred_train = nn_model.predict(X_train_scaled)
y_pred_val = nn_model.predict(X_val_scaled)
y_pred_test = nn_model.predict(X_test_scaled)

# Evaluate performance on the training set
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
r2_train = r2_score(y_train, y_pred_train)

# Evaluate performance on the validation set
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2_val = r2_score(y_val, y_pred_val)

# Evaluate performance on the test set
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)

# Print the results
print("Neural Network Model Performance:")
print(f"  Training Set: RMSE = {rmse_train:.2f}, R-squared = {r2_train:.2f}")
print(f"  Validation Set: RMSE = {rmse_val:.2f}, R-squared = {r2_val:.2f}")
print(f"  Test Set: RMSE = {rmse_test:.2f}, R-squared = {r2_test:.2f}")

Training and validation look good.  But Test does not.  Let's look at the residuals to see what is happening.  

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate residuals for all three sets
residuals_train = y_train - y_pred_train
residuals_val = y_val - y_pred_val
residuals_test = y_test - y_pred_test

# Create a figure and a set of subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5)) # 1 row, 3 columns

# Plot the distribution of residuals for the training set
sns.histplot(residuals_train, kde=True, ax=axes[0])
axes[0].set_title('Distribution of Residuals (Training Set)')
axes[0].set_xlabel('Residuals (Actual Price - Predicted Price)')
axes[0].set_ylabel('Frequency')

# Plot the distribution of residuals for the validation set
sns.histplot(residuals_val, kde=True, ax=axes[1])
axes[1].set_title('Distribution of Residuals (Validation Set)')
axes[1].set_xlabel('Residuals (Actual Price - Predicted Price)')
axes[1].set_ylabel('Frequency')

# Plot the distribution of residuals for the test set
sns.histplot(residuals_test, kde=True, ax=axes[2])
axes[2].set_title('Distribution of Residuals (Test Set)')
axes[2].set_xlabel('Residuals (Actual Price - Predicted Price)')
axes[2].set_ylabel('Frequency')

# Adjust layout to prevent overlapping titles/labels
plt.tight_layout()
plt.show()

# Also display summary statistics of residuals for all three sets
print("\nSummary Statistics of Residuals (Training Set):")
print(residuals_train.describe())

print("\nSummary Statistics of Residuals (Validation Set):")
print(residuals_val.describe())

print("\nSummary Statistics of Residuals (Test Set):")
print(residuals_test.describe())

And the actual by predicted plots.  Both these and the residuals show one car in Test where the prediction was very far off.  This is probably skewing our results.  

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure and a set of subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5)) # 1 row, 3 columns

# Plot actual vs. predicted for the training set
sns.scatterplot(x=y_train, y=y_pred_train, ax=axes[0])
axes[0].set_title('Actual vs. Predicted Price (Training Set)')
axes[0].set_xlabel('Actual Price')
axes[0].set_ylabel('Predicted Price')
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--') # Add a diagonal line

# Plot actual vs. predicted for the validation set
sns.scatterplot(x=y_val, y=y_pred_val, ax=axes[1])
axes[1].set_title('Actual vs. Predicted Price (Validation Set)')
axes[1].set_xlabel('Actual Price')
axes[1].set_ylabel('Predicted Price')
axes[1].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--') # Add a diagonal line


# Plot actual vs. predicted for the test set
sns.scatterplot(x=y_test, y=y_pred_test, ax=axes[2])
axes[2].set_title('Actual vs. Predicted Price (Test Set)')
axes[2].set_xlabel('Actual Price')
axes[2].set_ylabel('Predicted Price')
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--') # Add a diagonal line


# Adjust layout to prevent overlapping titles/labels
plt.tight_layout()
plt.show()

#Ensemble Models

In order to form an ensemble model, we need at least two models to combine.  So let's build a simple linear regression to combine with our NN model.

In [ ]:
from sklearn.linear_model import LinearRegression

# Define the features for the linear regression model
linear_features = ['Age', 'Mileage', 'Horse Power', 'Weight']

# Create and train the linear regression model
lr_model = LinearRegression()
lr_model.fit(X_train[linear_features], y_train)

# Print the coefficients
print("Linear Regression Model Coefficients:")
for feature, coef in zip(linear_features, lr_model.coef_):
    print(f"  {feature}: {coef:.2f}")
print(f"  Intercept: {lr_model.intercept_:.2f}")

Now let's check its performance.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Make predictions on all three sets using the linear regression model
y_pred_train_lr = lr_model.predict(X_train[linear_features])
y_pred_val_lr = lr_model.predict(X_val[linear_features])
y_pred_test_lr = lr_model.predict(X_test[linear_features])

# Evaluate performance on the training set
rmse_train_lr = np.sqrt(mean_squared_error(y_train, y_pred_train_lr))
r2_train_lr = r2_score(y_train, y_pred_train_lr)

# Evaluate performance on the validation set
rmse_val_lr = np.sqrt(mean_squared_error(y_val, y_pred_val_lr))
r2_val_lr = r2_score(y_val, y_pred_val_lr)

# Evaluate performance on the test set
rmse_test_lr = np.sqrt(mean_squared_error(y_test, y_pred_test_lr))
r2_test_lr = r2_score(y_test, y_pred_test_lr)

# Print the results
print("Linear Regression Model Performance:")
print(f"  Training Set: RMSE = {rmse_train_lr:.2f}, R-squared = {r2_train_lr:.2f}")
print(f"  Validation Set: RMSE = {rmse_val_lr:.2f}, R-squared = {r2_val_lr:.2f}")
print(f"  Test Set: RMSE = {rmse_test_lr:.2f}, R-squared = {r2_test_lr:.2f}")

Now that we have two models, we can do simple model averaging to form a new ensemble model.

In [ ]:
# Average the predictions from the two models
y_pred_train_ensemble = (y_pred_train + y_pred_train_lr) / 2
y_pred_val_ensemble = (y_pred_val + y_pred_val_lr) / 2
y_pred_test_ensemble = (y_pred_test + y_pred_test_lr) / 2

# Evaluate the ensemble model on the training set
rmse_train_ensemble = np.sqrt(mean_squared_error(y_train, y_pred_train_ensemble))
r2_train_ensemble = r2_score(y_train, y_pred_train_ensemble)

# Evaluate the ensemble model on the validation set
rmse_val_ensemble = np.sqrt(mean_squared_error(y_val, y_pred_val_ensemble))
r2_val_ensemble = r2_score(y_val, y_pred_val_ensemble)

# Evaluate the ensemble model on the test set
rmse_test_ensemble = np.sqrt(mean_squared_error(y_test, y_pred_test_ensemble))
r2_test_ensemble = r2_score(y_test, y_pred_test_ensemble)

# Print the results
print("Ensemble Model Performance (Averaging):")
print(f"  Training Set: RMSE = {rmse_train_ensemble:.2f}, R-squared = {r2_train_ensemble:.2f}")
print(f"  Validation Set: RMSE = {rmse_val_ensemble:.2f}, R-squared = {r2_val_ensemble:.2f}")
print(f"  Test Set: RMSE = {rmse_test_ensemble:.2f}, R-squared = {r2_test_ensemble:.2f}")

We could scroll up and down through the notebook to compare results, but a single model comparison table is much better.

In [ ]:
# Create a dictionary to store the performance metrics
performance_data = {
    'Model': ['Neural Network', 'Linear Regression', 'Ensemble (Averaging)'],
    'Train RMSE': [rmse_train, rmse_train_lr, rmse_train_ensemble],
    'Train R-squared': [r2_train, r2_train_lr, r2_train_ensemble],
    'Validation RMSE': [rmse_val, rmse_val_lr, rmse_val_ensemble],
    'Validation R-squared': [r2_val, r2_val_lr, r2_val_ensemble],
    'Test RMSE': [rmse_test, rmse_test_lr, rmse_test_ensemble],
    'Test R-squared': [r2_test, r2_test_lr, r2_test_ensemble]
}

# Create a pandas DataFrame from the dictionary
performance_df = pd.DataFrame(performance_data)

# Display the DataFrame
print("Model Performance Comparison:")
display(performance_df.round(2))

We can see that the ensemble outperforms the linear regression, but doesn't beat the NN.  So NN is the best!